In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = (
    SparkSession.builder.appName("iceberg-hands-on")
    # pulls the Iceberg jar automatically from Maven — no manual download
    .config("spark.jars.packages",
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.9.1")
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.local.type", "hadoop")
    .config("spark.sql.catalog.local.warehouse", "/home/iceberg/warehouse")
    .getOrCreate()
)
spark

26/09/17 11:32:39 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [7]:
spark.conf.get("spark.sql.defaultCatalog")
spark.conf.get("spark.sql.catalog.local")

'org.apache.iceberg.spark.SparkCatalog'

In [5]:
%%sql
CREATE NAMESPACE IF NOT EXISTS local.sales

++
||
++
++

In [6]:
%%sql
SHOW NAMESPACES IN local

namespace
sales


In [8]:
%%sql
CREATE TABLE local.sales.orders (
    order_id BIGINT,
    customer STRING,
    product STRING,
    amount DOUBLE,
    order_ts TIMESTAMP
  )USING iceberg PARTITIONED BY (days(order_ts))

++
||
++
++

In [9]:
%%sql
INSERT INTO local.sales.orders VALUES
 (1, 'ravi',  'keyboard', 2500.0,  TIMESTAMP '2026-07-01 09:15:00'),
 (2, 'sneha', 'monitor',  12000.0, TIMESTAMP '2026-07-01 14:30:00'),
 (3, 'arjun', 'mouse',    800.0,   TIMESTAMP '2026-07-02 10:05:00'),
 (4, 'priya', 'laptop',   65000.0, TIMESTAMP '2026-07-02 18:45:00'),
 (5, 'ravi',  'webcam',   3200.0,  TIMESTAMP '2026-07-03 11:20:00');

++
||
++
++

In [10]:
%%sql
SELECT * FROM local.sales.orders

order_id,customer,product,amount,order_ts
3,arjun,mouse,800.0,2026-07-02 10:05:00
4,priya,laptop,65000.0,2026-07-02 18:45:00
5,ravi,webcam,3200.0,2026-07-03 11:20:00
1,ravi,keyboard,2500.0,2026-07-01 09:15:00
2,sneha,monitor,12000.0,2026-07-01 14:30:00


In [11]:
%%sql
SELECT * FROM local.sales.orders.files

content,file_path,file_format,spec_id,partition,record_count,file_size_in_bytes,column_sizes,value_counts,null_value_counts,nan_value_counts,lower_bounds,upper_bounds,key_metadata,split_offsets,equality_ids,sort_order_id,referenced_data_file,content_offset,content_size_in_bytes,readable_metrics
0,/home/iceberg/warehouse/sales/orders/data/order_ts_day=2026-07-02/00000-5-2011e1c5-bff7-4938-9eb0-8c746b50d512-0-00002.parquet,PARQUET,0,"Row(order_ts_day=datetime.date(2026, 7, 2))",2,1564,"{1: 51, 2: 53, 3: 54, 4: 51, 5: 51}","{1: 2, 2: 2, 3: 2, 4: 2, 5: 2}","{1: 0, 2: 0, 3: 0, 4: 0, 5: 0}",{4: 0},"{1: bytearray(b'\x03\x00\x00\x00\x00\x00\x00\x00'), 2: bytearray(b'arjun'), 3: bytearray(b'laptop'), 4: bytearray(b'\x00\x00\x00\x00\x00\x00\x89@'), 5: bytearray(b'\x00\x8b\xe4\xf0\x9dU\x06\x00')}","{1: bytearray(b'\x04\x00\x00\x00\x00\x00\x00\x00'), 2: bytearray(b'priya'), 3: bytearray(b'mouse'), 4: bytearray(b'\x00\x00\x00\x00\x00\xbd\xef@'), 5: bytearray(b'\x00\xc3\x8e4\xa5U\x06\x00')}",None,[4],None,0,None,None,None,"Row(amount=Row(column_size=51, value_count=2, null_value_count=0, nan_value_count=0, lower_bound=800.0, upper_bound=65000.0), customer=Row(column_size=53, value_count=2, null_value_count=0, nan_value_count=None, lower_bound='arjun', upper_bound='priya'), order_id=Row(column_size=51, value_count=2, null_value_count=0, nan_value_count=None, lower_bound=3, upper_bound=4), order_ts=Row(column_size=51, value_count=2, null_value_count=0, nan_value_count=None, lower_bound=datetime.datetime(2026, 7, 2, 10, 5), upper_bound=datetime.datetime(2026, 7, 2, 18, 45)), product=Row(column_size=54, value_count=2, null_value_count=0, nan_value_count=None, lower_bound='laptop', upper_bound='mouse'))"
0,/home/iceberg/warehouse/sales/orders/data/order_ts_day=2026-07-03/00000-5-2011e1c5-bff7-4938-9eb0-8c746b50d512-0-00003.parquet,PARQUET,0,"Row(order_ts_day=datetime.date(2026, 7, 3))",1,1547,"{1: 43, 2: 43, 3: 45, 4: 43, 5: 43}","{1: 1, 2: 1, 3: 1, 4: 1, 5: 1}","{1: 0, 2: 0, 3: 0, 4: 0, 5: 0}",{4: 0},"{1: bytearray(b'\x05\x00\x00\x00\x00\x00\x00\x00'), 2: bytearray(b'ravi'), 3: bytearray(b'webcam'), 4: bytearray(b'\x00\x00\x00\x00\x00\x00\xa9@'), 5: bytearray(b'\x00x\xf4\x1a\xb3U\x06\x00')}","{1: bytearray(b'\x05\x00\x00\x00\x00\x00\x00\x00'), 2: bytearray(b'ravi'), 3: bytearray(b'webcam'), 4: bytearray(b'\x00\x00\x00\x00\x00\x00\xa9@'), 5: bytearray(b'\x00x\xf4\x1a\xb3U\x06\x00')}",None,[4],None,0,None,None,None,"Row(amount=Row(column_size=43, value_count=1, null_value_count=0, nan_value_count=0, lower_bound=3200.0, upper_bound=3200.0), customer=Row(column_size=43, value_count=1, null_value_count=0, nan_value_count=None, lower_bound='ravi', upper_bound='ravi'), order_id=Row(column_size=43, value_count=1, null_value_count=0, nan_value_count=None, lower_bound=5, upper_bound=5), order_ts=Row(column_size=43, value_count=1, null_value_count=0, nan_value_count=None, lower_bound=datetime.datetime(2026, 7, 3, 11, 20), upper_bound=datetime.datetime(2026, 7, 3, 11, 20)), product=Row(column_size=45, value_count=1, null_value_count=0, nan_value_count=None, lower_bound='webcam', upper_bound='webcam'))"
0,/home/iceberg/warehouse/sales/orders/data/order_ts_day=2026-07-01/00000-5-2011e1c5-bff7-4938-9eb0-8c746b50d512-0-00001.parquet,PARQUET,0,"Row(order_ts_day=datetime.date(2026, 7, 1))",2,1573,"{1: 51, 2: 52, 3: 58, 4: 51, 5: 51}","{1: 2, 2: 2, 3: 2, 4: 2, 5: 2}","{1: 0, 2: 0, 3: 0, 4: 0, 5: 0}",{4: 0},"{1: bytearray(b'\x01\x00\x00\x00\x00\x00\x00\x00'), 2: bytearray(b'ravi'), 3: bytearray(b'keyboard'), 4: bytearray(b'\x00\x00\x00\x00\x00\x88\xa3@'), 5: bytearray(b'\x00\xcd< \x89U\x06\x00')}","{1: bytearray(b'\x02\x00\x00\x00\x00\x00\x00\x00'), 2: bytearray(b'sneha'), 3: bytearray(b'monitor'), 4: bytearray(b'\x00\x00\x00\x00\x00p\xc7@'), 5: bytearray(b'\x00\xea\xc3\x86\x8dU\x06\x00')}",None,[4],None,0,None,None,None,"Row(amount=Row(column_size=51, value_count=2, null_value_count=0, nan_value_count=0, lower_bound=2500.0, upper_bound=12000.0), customer=Row(colu

## Hidden Partitioning

In [20]:
%%sql

SELECT order_id, customer, product, amount
FROM local.sales.orders
WHERE order_ts BETWEEN TIMESTAMP '2026-07-02 00:00:00'
                   AND TIMESTAMP '2026-07-02 23:59:59';

order_id,customer,product,amount
3,arjun,mouse,800.0
4,priya,laptop,65000.0


In [13]:
%%sql
SELECT partition, record_count, file_count
FROM local.sales.orders.partitions ORDER BY partition;

partition,record_count,file_count
"Row(order_ts_day=datetime.date(2026, 7, 1))",2,1
"Row(order_ts_day=datetime.date(2026, 7, 2))",2,1
"Row(order_ts_day=datetime.date(2026, 7, 3))",1,1


## Schema Evolution

In [19]:
%%sql
DESCRIBE EXTENDED local.sales.orders;

col_name,data_type,comment
order_id,bigint,None
customer,string,None
product,string,None
amount,double,None
order_ts,timestamp,None
,,
# Partitioning,,
Part 0,days(order_ts),
,,
# Metadata Columns,,


In [16]:
%%sql
SELECT system.iceberg_version();

"staticinvoke(class org.apache.iceberg.spark.functions.IcebergVersionFunction$IcebergVersionFunctionImpl, StringType, invoke, false, false, true)"
1.8.1


In [21]:
%%sql
SELECT * FROM local.sales.orders;

order_id,customer,product,amount,order_ts
3,arjun,mouse,800.0,2026-07-02 10:05:00
4,priya,laptop,65000.0,2026-07-02 18:45:00
5,ravi,webcam,3200.0,2026-07-03 11:20:00
1,ravi,keyboard,2500.0,2026-07-01 09:15:00
2,sneha,monitor,12000.0,2026-07-01 14:30:00


In [22]:
%%sql
ALTER TABLE local.sales.orders ADD COLUMN discount DOUBLE;

++
||
++
++

In [23]:
%%sql
DESCRIBE local.sales.orders;

col_name,data_type,comment
order_id,bigint,None
customer,string,None
product,string,None
amount,double,None
order_ts,timestamp,None
discount,double,None
,,
# Partitioning,,
Part 0,days(order_ts),


In [24]:
%%sql
ALTER TABLE local.sales.orders RENAME COLUMN amount to total_amount;

++
||
++
++

In [27]:
%%sql
INSERT INTO local.sales.orders VALUES
 (6, 'meera', 'headset', 4500.0, TIMESTAMP '2026-07-03 16:10:00', 500.0);

++
||
++
++

In [29]:
%%sql
SELECT * FROM local.sales.orders ORDER BY order_id

order_id,customer,product,total_amount,order_ts,discount
1,ravi,keyboard,2500.0,2026-07-01 09:15:00,None
2,sneha,monitor,12000.0,2026-07-01 14:30:00,None
3,arjun,mouse,800.0,2026-07-02 10:05:00,None
4,priya,laptop,65000.0,2026-07-02 18:45:00,None
5,ravi,webcam,3200.0,2026-07-03 11:20:00,None
6,meera,headset,4500.0,2026-07-03 16:10:00,500.0


## Snapshots: The table format that remembers everything

In [30]:
%%sql

SELECT snapshot_id, committed_at, operation,
    summary['added-records'] AS added
FROM local.sales.orders.snapshots ORDER BY committed_at;

snapshot_id,committed_at,operation,added
7981229231077807380,2026-09-17 11:39:54.403000,append,5
4464673354906844485,2026-09-17 12:45:27.335000,append,1


In [31]:
%%sql
DESCRIBE local.sales.orders.snapshots

col_name,data_type,comment
committed_at,timestamp,None
snapshot_id,bigint,None
parent_id,bigint,None
operation,string,None
manifest_list,string,None
summary,"map<string,string>",None


## Time Travel

In [32]:
%%sql
SELECT snapshot_id FROM local.sales.orders.snapshots
ORDER BY committed_at DESC LIMIT 1

snapshot_id
4464673354906844485


In [33]:
%%sql
-- query table as it was after FIRST insert
-- query using snapshot id before FIRST insert
SELECT *
FROM local.sales.orders VERSION AS OF 7981229231077807380;

order_id,customer,product,amount,order_ts
3,arjun,mouse,800.0,2026-07-02 10:05:00
4,priya,laptop,65000.0,2026-07-02 18:45:00
5,ravi,webcam,3200.0,2026-07-03 11:20:00
1,ravi,keyboard,2500.0,2026-07-01 09:15:00
2,sneha,monitor,12000.0,2026-07-01 14:30:00


In [36]:
%%sql
SELECT *
FROM local.sales.orders TIMESTAMP AS OF '2026-09-17 11:50:27.335000';

order_id,customer,product,amount,order_ts
3,arjun,mouse,800.0,2026-07-02 10:05:00
4,priya,laptop,65000.0,2026-07-02 18:45:00
5,ravi,webcam,3200.0,2026-07-03 11:20:00
1,ravi,keyboard,2500.0,2026-07-01 09:15:00
2,sneha,monitor,12000.0,2026-07-01 14:30:00


## Recovering table state using snapshot 
Example: a wrong query to set some values unintended


In [38]:
%%sql
SELECT * FROM local.sales.orders ORDER BY order_id;

order_id,customer,product,total_amount,order_ts,discount
1,ravi,keyboard,2500.0,2026-07-01 09:15:00,None
2,sneha,monitor,12000.0,2026-07-01 14:30:00,None
3,arjun,mouse,800.0,2026-07-02 10:05:00,None
4,priya,laptop,65000.0,2026-07-02 18:45:00,None
5,ravi,webcam,3200.0,2026-07-03 11:20:00,None
6,meera,headset,4500.0,2026-07-03 16:10:00,500.0


In [39]:
%%sql
UPDATE local.sales.orders SET total_amount=0 WHERE order_id=4

++
||
++
++

In [40]:
%%sql
SELECT * FROM local.sales.orders ORDER BY order_id;

order_id,customer,product,total_amount,order_ts,discount
1,ravi,keyboard,2500.0,2026-07-01 09:15:00,None
2,sneha,monitor,12000.0,2026-07-01 14:30:00,None
3,arjun,mouse,800.0,2026-07-02 10:05:00,None
4,priya,laptop,0.0,2026-07-02 18:45:00,None
5,ravi,webcam,3200.0,2026-07-03 11:20:00,None
6,meera,headset,4500.0,2026-07-03 16:10:00,500.0


In [41]:
%%sql
SELECT *
FROM local.sales.orders.snapshots ORDER BY committed_at;

committed_at,snapshot_id,parent_id,operation,manifest_list,summary
2026-09-17 11:39:54.403000,7981229231077807380,None,append,/home/iceberg/warehouse/sales/orders/metadata/snap-7981229231077807380-1-0a5a08b4-645a-4886-b8d9-5ebcb7ecbd1f.avro,"{'engine-version': '3.5.5', 'added-data-files': '3', 'total-equality-deletes': '0', 'app-id': 'local-1789644657125', 'added-records': '5', 'total-records': '5', 'spark.app.id': 'local-1789644657125', 'changed-partition-count': '3', 'engine-name': 'spark', 'total-position-deletes': '0', 'added-files-size': '4684', 'total-delete-files': '0', 'iceberg-version': 'Apache Iceberg 1.8.1 (commit 9ce0fcf0af7becf25ad9fc996c3bad2afdcfd33d)', 'total-files-size': '4684', 'total-data-files': '3'}"
2026-09-17 12:45:27.335000,4464673354906844485,7981229231077807380,append,/home/iceberg/warehouse/sales/orders/metadata/snap-4464673354906844485-1-d9c29c9e-ba43-401b-8c76-e28eb13e9337.avro,"{'engine-version': '3.5.5', 'added-data-files': '1', 'total-equality-deletes': '0', 'app-id': 'local-1789644657125', 'added-records': '1', 'total-records': '6', 'spark.app.id': 'local-1789644657125', 'changed-partition-count': '1', 'engine-name': 'spark', 'total-position-deletes': '0', 'added-files-size': '1853', 'total-delete-files': '0', 'iceberg-version': 'Apache Iceberg 1.8.1 (commit 9ce0fcf0af7becf25ad9fc996c3bad2afdcfd33d)', 'total-files-size': '6537', 'total-data-files': '4'}"
2026-09-17 13:33:41.678000,885988385870046479,4464673354906844485,overwrite,/home/iceberg/warehouse/sales/orders/metadata/snap-885988385870046479-1-432db0f7-fe58-4a75-99a2-d4a6535390df.avro,"{'engine-version': '3.5.5', 'added-data-files': '1', 'total-equality-deletes': '0', 'app-id': 'local-1789644657125', 'added-records': '2', 'deleted-data-files': '1', 'deleted-records': '2', 'total-records': '6', 'spark.app.id': 'local-1789644657125', 'removed-files-size': '1564', 'changed-partition-count': '1', 'engine-name': 'spark', 'total-position-deletes': '0', 'added-files-size': '1852', 'total-delete-files': '0', 'iceberg-version': 'Apache Iceberg 1.8.1 (commit 9ce0fcf0af7becf25ad9fc996c3bad2afdcfd33d)', 'total-files-size': '6825', 'total-data-files': '4'}"


In [42]:
%%sql
CALL local.system.rollback_to_snapshot('sales.orders', 4464673354906844485)

previous_snapshot_id,current_snapshot_id
885988385870046479,4464673354906844485


In [44]:
%%sql
-- Checking if rollback worked right
SELECT * FROM local.sales.orders ORDER BY order_id;

order_id,customer,product,total_amount,order_ts,discount
1,ravi,keyboard,2500.0,2026-07-01 09:15:00,None
2,sneha,monitor,12000.0,2026-07-01 14:30:00,None
3,arjun,mouse,800.0,2026-07-02 10:05:00,None
4,priya,laptop,65000.0,2026-07-02 18:45:00,None
5,ravi,webcam,3200.0,2026-07-03 11:20:00,None
6,meera,headset,4500.0,2026-07-03 16:10:00,500.0


In [45]:
%%sql
SELECT *
FROM local.sales.orders.snapshots ORDER BY committed_at;

committed_at,snapshot_id,parent_id,operation,manifest_list,summary
2026-09-17 11:39:54.403000,7981229231077807380,None,append,/home/iceberg/warehouse/sales/orders/metadata/snap-7981229231077807380-1-0a5a08b4-645a-4886-b8d9-5ebcb7ecbd1f.avro,"{'engine-version': '3.5.5', 'added-data-files': '3', 'total-equality-deletes': '0', 'app-id': 'local-1789644657125', 'added-records': '5', 'total-records': '5', 'spark.app.id': 'local-1789644657125', 'changed-partition-count': '3', 'engine-name': 'spark', 'total-position-deletes': '0', 'added-files-size': '4684', 'total-delete-files': '0', 'iceberg-version': 'Apache Iceberg 1.8.1 (commit 9ce0fcf0af7becf25ad9fc996c3bad2afdcfd33d)', 'total-files-size': '4684', 'total-data-files': '3'}"
2026-09-17 12:45:27.335000,4464673354906844485,7981229231077807380,append,/home/iceberg/warehouse/sales/orders/metadata/snap-4464673354906844485-1-d9c29c9e-ba43-401b-8c76-e28eb13e9337.avro,"{'engine-version': '3.5.5', 'added-data-files': '1', 'total-equality-deletes': '0', 'app-id': 'local-1789644657125', 'added-records': '1', 'total-records': '6', 'spark.app.id': 'local-1789644657125', 'changed-partition-count': '1', 'engine-name': 'spark', 'total-position-deletes': '0', 'added-files-size': '1853', 'total-delete-files': '0', 'iceberg-version': 'Apache Iceberg 1.8.1 (commit 9ce0fcf0af7becf25ad9fc996c3bad2afdcfd33d)', 'total-files-size': '6537', 'total-data-files': '4'}"
2026-09-17 13:33:41.678000,885988385870046479,4464673354906844485,overwrite,/home/iceberg/warehouse/sales/orders/metadata/snap-885988385870046479-1-432db0f7-fe58-4a75-99a2-d4a6535390df.avro,"{'engine-version': '3.5.5', 'added-data-files': '1', 'total-equality-deletes': '0', 'app-id': 'local-1789644657125', 'added-records': '2', 'deleted-data-files': '1', 'deleted-records': '2', 'total-records': '6', 'spark.app.id': 'local-1789644657125', 'removed-files-size': '1564', 'changed-partition-count': '1', 'engine-name': 'spark', 'total-position-deletes': '0', 'added-files-size': '1852', 'total-delete-files': '0', 'iceberg-version': 'Apache Iceberg 1.8.1 (commit 9ce0fcf0af7becf25ad9fc996c3bad2afdcfd33d)', 'total-files-size': '6825', 'total-data-files': '4'}"


In [46]:
%%sql
-- Checking history but it doesnt show operation and snapshots doesnt show me
SELECT * FROM local.sales.orders.history;

made_current_at,snapshot_id,parent_id,is_current_ancestor
2026-09-17 11:39:54.403000,7981229231077807380,None,True
2026-09-17 12:45:27.335000,4464673354906844485,7981229231077807380,True
2026-09-17 13:33:41.678000,885988385870046479,4464673354906844485,False
2026-09-17 13:41:54.914000,4464673354906844485,7981229231077807380,True


In [56]:
%%sql
-- getting the history better along with operation that created the snapshots
-- diff between made_current_at and committed_at for current snapshot tells us about the rollback
select
    h.made_current_at,
    s.committed_at,
    s.operation,
    h.snapshot_id,
    h.is_current_ancestor
from local.sales.orders.history h
join local.sales.orders.snapshots s
  on h.snapshot_id = s.snapshot_id
order by made_current_at;

made_current_at,committed_at,operation,snapshot_id,is_current_ancestor
2026-09-17 11:39:54.403000,2026-09-17 11:39:54.403000,append,7981229231077807380,True
2026-09-17 12:45:27.335000,2026-09-17 12:45:27.335000,append,4464673354906844485,True
2026-09-17 13:33:41.678000,2026-09-17 13:33:41.678000,overwrite,885988385870046479,False
2026-09-17 13:41:54.914000,2026-09-17 12:45:27.335000,append,4464673354906844485,True


## Inspecting various metadata files for learning

In [54]:
%%sql
-- Checking for metadata log entries
SELECT * FROM local.sales.orders.metadata_log_entries;

timestamp,file,latest_snapshot_id,latest_schema_id,latest_sequence_number
2026-09-17 11:39:20.791000,/home/iceberg/warehouse/sales/orders/metadata/v1.metadata.json,None,None,None
2026-09-17 11:39:54.403000,/home/iceberg/warehouse/sales/orders/metadata/v2.metadata.json,7981229231077807380,0,1
2026-09-17 12:41:44.397000,/home/iceberg/warehouse/sales/orders/metadata/v3.metadata.json,7981229231077807380,0,1
2026-09-17 12:42:48.097000,/home/iceberg/warehouse/sales/orders/metadata/v4.metadata.json,7981229231077807380,0,1
2026-09-17 12:45:27.335000,/home/iceberg/warehouse/sales/orders/metadata/v5.metadata.json,4464673354906844485,2,2
2026-09-17 13:33:41.678000,/home/iceberg/warehouse/sales/orders/metadata/v6.metadata.json,885988385870046479,2,3
2026-09-17 13:41:54.914000,/home/iceberg/warehouse/sales/orders/metadata/v7.metadata.json,4464673354906844485,2,2


In [64]:
%%sql
SELECT
    s.operation,
    h.snapshot_id,
    h.is_current_ancestor,
    am.added_data_files_count,
    am.deleted_data_files_count
FROM local.sales.orders.snapshots s
JOIN local.sales.orders.all_manifests am
    ON am.reference_snapshot_id = s.snapshot_id 
JOIN local.sales.orders.history h
    ON s.snapshot_id = h.snapshot_id
order by h.made_current_at;

operation,snapshot_id,is_current_ancestor,added_data_files_count,deleted_data_files_count
append,7981229231077807380,True,3,0
append,4464673354906844485,True,3,0
append,4464673354906844485,True,1,0
overwrite,885988385870046479,False,0,1
overwrite,885988385870046479,False,1,0
overwrite,885988385870046479,False,1,0
append,4464673354906844485,True,3,0
append,4464673354906844485,True,1,0


## Upserts with Iceberg

In [67]:
%%sql
-- checking table before upsert
SELECT * FROM local.sales.orders;

order_id,customer,product,total_amount,order_ts,discount
3,arjun,mouse,800.0,2026-07-02 10:05:00,None
4,priya,laptop,65000.0,2026-07-02 18:45:00,None
5,ravi,webcam,3200.0,2026-07-03 11:20:00,None
1,ravi,keyboard,2500.0,2026-07-01 09:15:00,None
2,sneha,monitor,12000.0,2026-07-01 14:30:00,None
6,meera,headset,4500.0,2026-07-03 16:10:00,500.0


In [66]:
%%sql
CREATE OR REPLACE TEMPORARY VIEW order_updates AS
SELECT * FROM VALUES
     (5, 'ravi',  'webcam', 2900.0, TIMESTAMP '2026-07-03 11:20:00', 300.0),
     (7, 'kiran', 'ssd',    5500.0, TIMESTAMP '2026-07-04 09:00:00', 0.0)
AS t(order_id, customer, product, total_amount, order_ts, discount);

++
||
++
++

In [68]:
%%sql
MERGE INTO local.sales.orders o
USING order_updates u ON o.order_id = u.order_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

++
||
++
++

In [69]:
%%sql
SELECT * FROM local.sales.orders;

order_id,customer,product,total_amount,order_ts,discount
6,meera,headset,4500.0,2026-07-03 16:10:00,500.0
5,ravi,webcam,2900.0,2026-07-03 11:20:00,300.0
7,kiran,ssd,5500.0,2026-07-04 09:00:00,0.0
3,arjun,mouse,800.0,2026-07-02 10:05:00,None
4,priya,laptop,65000.0,2026-07-02 18:45:00,None
1,ravi,keyboard,2500.0,2026-07-01 09:15:00,None
2,sneha,monitor,12000.0,2026-07-01 14:30:00,None


In [72]:
%%sql
SELECT snapshot_id, committed_at, operation
FROM local.sales.orders.snapshots ORDER BY committed_at;

snapshot_id,committed_at,operation
7981229231077807380,2026-09-17 11:39:54.403000,append
4464673354906844485,2026-09-17 12:45:27.335000,append
885988385870046479,2026-09-17 13:33:41.678000,overwrite
9060840941276147221,2026-09-17 16:58:32.519000,overwrite


## Partition evolution

In [73]:
%%sql
SELECT * FROM local.sales.orders;

order_id,customer,product,total_amount,order_ts,discount
3,arjun,mouse,800.0,2026-07-02 10:05:00,None
4,priya,laptop,65000.0,2026-07-02 18:45:00,None
1,ravi,keyboard,2500.0,2026-07-01 09:15:00,None
2,sneha,monitor,12000.0,2026-07-01 14:30:00,None
6,meera,headset,4500.0,2026-07-03 16:10:00,500.0
5,ravi,webcam,2900.0,2026-07-03 11:20:00,300.0
7,kiran,ssd,5500.0,2026-07-04 09:00:00,0.0


In [75]:
%%sql
ALTER TABLE local.sales.orders ADD PARTITION FIELD bucket(4, customer) 

++
||
++
++

In [76]:
%%sql
INSERT INTO local.sales.orders VALUES
(8, 'anita', 'gpu', 95000.0, TIMESTAMP '2026-07-05 12:00:00', 0.0);

++
||
++
++

In [83]:
# why bucket no 3 was chosen for above record
# awesome murmur3
import mmh3
(mmh3.hash("anita", 0) & 0x7fffffff) % 4

3

In [78]:
%%sql
-- notice partition evolution applied only for a new record after
SELECT partition, record_count FROM local.sales.orders.partitions ORDER BY partition;

partition,record_count
"Row(order_ts_day=datetime.date(2026, 7, 1), customer_bucket_4=None)",2
"Row(order_ts_day=datetime.date(2026, 7, 2), customer_bucket_4=None)",2
"Row(order_ts_day=datetime.date(2026, 7, 3), customer_bucket_4=None)",2
"Row(order_ts_day=datetime.date(2026, 7, 4), customer_bucket_4=None)",1
"Row(order_ts_day=datetime.date(2026, 7, 5), customer_bucket_4=3)",1


In [80]:
%%sql
SELECT * FROM local.sales.orders ORDER BY order_ts;

order_id,customer,product,total_amount,order_ts,discount
1,ravi,keyboard,2500.0,2026-07-01 09:15:00,None
2,sneha,monitor,12000.0,2026-07-01 14:30:00,None
3,arjun,mouse,800.0,2026-07-02 10:05:00,None
4,priya,laptop,65000.0,2026-07-02 18:45:00,None
5,ravi,webcam,2900.0,2026-07-03 11:20:00,300.0
6,meera,headset,4500.0,2026-07-03 16:10:00,500.0
7,kiran,ssd,5500.0,2026-07-04 09:00:00,0.0
8,anita,gpu,95000.0,2026-07-05 12:00:00,0.0


In [89]:
%%sql
-- Getting row to partition mapping just to confirm which partition the row belongs to
SELECT
  order_id,
  customer,
  order_ts,
  _spec_id,
  CAST(_partition AS STRING) AS part,
  regexp_extract(_file, '[^/]+$', 0) AS file_name
FROM local.sales.orders
ORDER BY order_ts

order_id,customer,order_ts,_spec_id,part,file_name
1,ravi,2026-07-01 09:15:00,0,"{2026-07-01, null}",00000-5-2011e1c5-bff7-4938-9eb0-8c746b50d512-0-00001.parquet
2,sneha,2026-07-01 14:30:00,0,"{2026-07-01, null}",00000-5-2011e1c5-bff7-4938-9eb0-8c746b50d512-0-00001.parquet
3,arjun,2026-07-02 10:05:00,0,"{2026-07-02, null}",00000-5-2011e1c5-bff7-4938-9eb0-8c746b50d512-0-00002.parquet
4,priya,2026-07-02 18:45:00,0,"{2026-07-02, null}",00000-5-2011e1c5-bff7-4938-9eb0-8c746b50d512-0-00002.parquet
5,ravi,2026-07-03 11:20:00,0,"{2026-07-03, null}",00000-66-04fc901c-7fda-49f5-bd0f-25687f414c0a-0-00001.parquet
6,meera,2026-07-03 16:10:00,0,"{2026-07-03, null}",00000-15-3af6fb52-552a-4df2-b4c3-777d46504251-0-00001.parquet
7,kiran,2026-07-04 09:00:00,0,"{2026-07-04, null}",00000-66-04fc901c-7fda-49f5-bd0f-25687f414c0a-0-00002.parquet
8,anita,2026-07-05 12:00:00,1,"{2026-07-05, 3}",00000-75-9c7d3aad-b819-4574-bf3b-2177637a69a7-0-00001.parquet


### NOTES

**_spec_id**
unique identifier for the partition spec active when a manifest or data file was written, 
enabling safe partition evolution over time partition was evolved, partition spec is 1 here for new insert above
                                                                                      
**_partition** The specific partition values associated with a data or delete file, 
            used for runtime partition pruning

                                                                                 
**_file / file_path** The absolute or relative URI path pointing directly to the target data file
                , commonly referenced inside positional delete files